In [ ]:
import os

import torch
import logging
logger = logging.getLogger("ignite.handlers.early_stopping.EarlyStopping")
logger.setLevel(logging.WARNING)


import torch

import numpy as np

import torch
import itertools
from tqdm import tqdm
import os
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed


import itertools


import torch
import os


import pandas as pd
from concurrent.futures import as_completed

from explanations import KRandom, Self
    
import logging
logging.getLogger().setLevel(logging.WARNING)



import logging
from concurrent.futures import ProcessPoolExecutor, as_completed
import torch
import multiprocessing
from tqdm import tqdm
import itertools
import pandas as pd
import traceback



logging.basicConfig(level=logging.ERROR, format='%(asctime)s [%(levelname)s] %(message)s')






multiprocessing.set_start_method('spawn', force=True)   
torch.manual_seed(42)
from load_experiment_data import (
train_dataset_name,
test_dataset_name,
train_dataset_split,
test_dataset_split,
load_data_and_estimators,
explanation_types,
linear_coders,
explanation_k,
explanation_seed
)
train_dataset, test_dataset, estimators = load_data_and_estimators()


In [ ]:
estimator_datainf = estimators[6]
assert (estimator_datainf.get_config_string() == 'DataInfEstimator: fast_implementation=True') and (estimator_datainf.model_path == './models/OLMo-2-0425-1B_tulu-3-sft-olmo-2-mixture-0225_lr0.0001_seed42')


In [ ]:
estimator_bm25 = estimators[7]
assert (estimator_bm25.get_config_string() == 'BM25Estimator: k1=1.5, b=0.75') and (estimator_bm25.model_path == './models/OLMo-2-0425-1B_tulu-3-sft-olmo-2-mixture-0225_lr0.0001_seed42')


In [ ]:
estimator_less = estimators[8]
assert (estimator_less.get_config_string() == 'LESSEstimator: normalize=True') and (estimator_less.model_path == './models/OLMo-2-0425-1B_tulu-3-sft-olmo-2-mixture-0225_lr0.0001_seed42')


In [ ]:
from explanations import TopKMostHelpful, Explanation,TopKMostHarmful, TopKMostInfluential,TopKLeastInfluential,DIVINEMostHelpful, DIVINEMostHarmful,FacilityLocationMostHarmful,FacilityLocationMostInfluential,FacilityLocationLeastInfluential,DIVINEMostInfluential,DIVINELeastInfluential,FacilityLocationMostHelpful#,AIDE
from fl_optimizers import LazyWeightedGreedy
from sklearn.preprocessing import RobustScaler, MinMaxScaler
from explanations import ImportanceLookupSelector
from apricot import FacilityLocationSelection, MixtureSelection, SumRedundancySelection, FeatureBasedSelection
from apricot.optimizers import LazyGreedy
from sklearn.metrics import pairwise_distances


In [ ]:
from explanations import AIDE


In [ ]:
from abc import ABC, abstractmethod
import numpy
from numba import njit



In [ ]:
from sklearn.preprocessing import RobustScaler, MinMaxScaler


In [ ]:
from influence_estimation.util import tokenize_dataset
from finetune import load_tokenizer

In [ ]:
tokenizer = load_tokenizer("allenai/OLMo-2-0425-1B")

In [ ]:
import time


In [ ]:
from sentence_transformers import SentenceTransformer
import torch

embedding_model = SentenceTransformer(
    "Qwen/Qwen3-Embedding-0.6B",
    tokenizer_kwargs={"padding_side": "left"},

)


In [ ]:
import matplotlib.colors as mcolors

base_colors = (
    (0.5529411764705883, 0.6274509803921569, 0.796078431372549),
    (0.4, 0.7607843137254902, 0.6470588235294118),
    (0.9882352941176471, 0.5529411764705883, 0.3843137254901961),
    
    (0.9058823529411765, 0.5411764705882353, 0.7647058823529411),
    (0.6509803921568628, 0.8470588235294118, 0.32941176470588235),
    (1.0, 0.8509803921568627, 0.1843137254901961),
    (0.8980392156862745, 0.7686274509803922, 0.5803921568627451),
    (0.7019607843137254, 0.7019607843137254, 0.7019607843137254)
)

def darken_color(rgb, factor=0.9):
    return tuple(factor * c for c in rgb)


base_colors = [darken_color(c, 0.9) for c in base_colors]

In [ ]:
import pandas as pd
linear_coder = "MSECoderProjUSimp"
df = pd.read_parquet("./results/scoring/merged.parquet")
df = df[df["linear_coder"] == linear_coder]

In [ ]:

import torch
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from scipy.spatial import ConvexHull
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.metrics.pairwise import cosine_distances

import seaborn as sns
from matplotlib.patches import Polygon
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize

from sklearn.manifold import locally_linear_embedding
from sklearn.manifold import MDS

def plot_single_grid(
    title,
    gradients_groundset,
    gradient_test_instance,
    embeddings_groundset,
    embedding_test_instance,
    methods=None,
    costs=None,
    vmax_percentile=95,
    n_rows=2,
    n_cols=8,
    figsize=(20, 6),
    k=None,
    test_idx=None,
    use_cosine=False,
    dim_reduction_method="tsne",
    cache={}    
):
    


    # Convert torch tensors to numpy
    def to_numpy(x):
        if isinstance(x, torch.Tensor):
            return x.detach().cpu().numpy()
        return x

    gradients_groundset = to_numpy(gradients_groundset)
    gradient_test_instance = to_numpy(gradient_test_instance)
    embeddings_groundset = to_numpy(embeddings_groundset)
    embedding_test_instance = to_numpy(embedding_test_instance)
    if costs is not None:
        costs = to_numpy(costs)



    

    def compute_tsne(embedding_type, test_idx, data, test_instance, cache, n_pca_components=50, perplexity=10):
        cache_key = ("tsne", embedding_type, test_idx, use_cosine)
        combined = np.vstack([data, test_instance.reshape(1, -1)])

        if cache_key in cache:
            xs, ys, pca, tsne, x_reduced = cache[cache_key]


        else: 
            if use_cosine:
                norms = np.linalg.norm(combined, axis=1, keepdims=True)
                combined = combined / (norms + 1e-10)  
            # PCA preprocessing
            pca = PCA(n_components=min(n_pca_components, combined.shape[1]))
            x_reduced = pca.fit_transform(combined)

            # t-SNE embedding
            tsne = TSNE(
                n_components=2,
                perplexity=perplexity,
                random_state=42,
                metric="cosine" if use_cosine else "euclidean",
                init='pca'
            )
            x_tsne = tsne.fit_transform(x_reduced)
            xs, ys = x_tsne[:, 0], x_tsne[:, 1]

    
            cache[cache_key] = (xs, ys, pca, tsne, x_reduced)

        return xs[:-1], ys[:-1], xs[-1], ys[-1], pca, x_reduced

    def compute_pca2d(embedding_type, test_idx, data, test_instance, cache):
        cache_key = ("pca2d", embedding_type, test_idx)
        combined = np.vstack([data, test_instance.reshape(1, -1)])

        if cache_key in cache:
            xs, ys, pca, x_reduced = cache[cache_key]

        else: 
            pca = PCA(n_components=2)
            x_reduced = pca.fit_transform(combined)
            xs, ys = x_reduced[:, 0], x_reduced[:, 1]
            cache[cache_key] = (xs, ys, pca, x_reduced)

        return xs[:-1], ys[:-1], xs[-1], ys[-1], pca, x_reduced

    def compute_mlle(
        embedding_type,
        test_idx,
        data,
        test_instance,
        cache,
        n_neighbors=12,
        lle_method="standard",
        n_pca_components=50,
        
    ):

        cache_key = ("lle", embedding_type, test_idx, use_cosine)

        combined = np.vstack([data, test_instance.reshape(1, -1)])

        if cache_key in cache:
            xs, ys, pca, lle_result, x_reduced = cache[cache_key]


        else: 
            pca = PCA(n_components=min(n_pca_components, combined.shape[1]))
            x_reduced = pca.fit_transform(combined)

            X_2d, err = locally_linear_embedding(
                x_reduced,
                n_neighbors=n_neighbors,
                n_components=2,
                method = 'modified'
            )

            xs, ys = X_2d[:, 0], X_2d[:, 1]
            lle_result = (X_2d, err)


            cache[cache_key] = (xs, ys, pca, lle_result, x_reduced)

        return xs[:-1], ys[:-1], xs[-1], ys[-1], pca, x_reduced
    
    def compute_mds(
        embedding_type,
        test_idx,
        data,
        test_instance,
        cache,  
        n_pca_components=50,
        n_mds_components=2,
        metric=True,            
        max_iter=300,
        random_state=42,
        n_init=1,
    ):
       
        cache_key = ("mds", embedding_type, test_idx, use_cosine)

        combined = np.vstack([data, test_instance.reshape(1, -1)])

        if cache_key in cache:
            xs, ys, pca, mds_model, x_reduced = cache[cache_key]
        else: 

    
            pca = PCA(n_components=min(n_pca_components, combined.shape[1]))
            x_reduced = pca.fit_transform(combined)

        
            mds = MDS(
                n_components=n_mds_components,
                metric=metric,
                max_iter=max_iter,
                n_init=n_init,
                random_state=42,
                dissimilarity=("cosine" if use_cosine else "euclidean"),   
                n_jobs=-1
            )

            X_2d = mds.fit_transform(x_reduced)
            xs, ys = X_2d[:, 0], X_2d[:, 1]

        
            mds_model = mds

          
            cache[cache_key] = (xs, ys, pca, mds_model, x_reduced)

        return xs[:-1], ys[:-1], xs[-1], ys[-1], pca, x_reduced
        
    
    dim_reduction_method_fn = None
    if dim_reduction_method == "tsne":
        dim_reduction_method_fn = compute_tsne
    if dim_reduction_method == "mlle":
            dim_reduction_method_fn = compute_mlle
    if dim_reduction_method == "mds":
            dim_reduction_method_fn = compute_mds
    if dim_reduction_method == "pca2d":
            dim_reduction_method_fn = compute_pca2d
    grad_x, grad_y, grad_test_x, grad_test_y, grad_pca, grad_reduced = dim_reduction_method_fn("grad", test_idx, gradients_groundset, gradient_test_instance, cache)
    emb_x, emb_y, emb_test_x, emb_test_y, emb_pca, emb_reduced = dim_reduction_method_fn("emb", test_idx, embeddings_groundset, embedding_test_instance, cache)

   
    markers = ["o", "s", "^", "D", "P", "X", "v"]
    method_to_marker = {name: markers[i % len(markers)] for i, (name, *_) in enumerate(methods)}
    method_to_color = {name: base_colors[1:][i % len(base_colors[1:])] for i, (name, *_) in enumerate(methods)}


    fig = plt.figure(figsize=figsize)
    gs = gridspec.GridSpec(n_rows, n_cols + 2, width_ratios=[0.15] + [1]*n_cols + [0.05], wspace=0, hspace=0)
    axes = []
    for r in range(n_rows):
        for c in range(n_cols):
            ax = fig.add_subplot(gs[r, c + 1]) 
            axes.append(ax)


    pred_gains = [m[5] for m in methods]
    min_gain = min(pred_gains)
    max_gain = max(pred_gains)
    for i, (name, _,method_grad, method_emb, estimator, pred_gain, label, underline) in enumerate(methods):
        method_grad = np.asarray(method_grad)
        method_emb = np.asarray(method_emb)

    
        method_grad_pca = grad_pca.transform(np.vstack([gradients_groundset, method_grad]))[-len(method_grad):]
        if use_cosine:
            dists_grad = cosine_distances(grad_reduced[:-1], method_grad_pca)
        else:
            dists_grad = np.linalg.norm(grad_reduced[:-1, None, :] - method_grad_pca[None, :, :], axis=2)

        idx_grad = np.argmin(dists_grad, axis=0)
        ax_grad = axes[i]
        
        

        vmax = np.percentile(costs, vmax_percentile)
        norm = Normalize(vmin=np.min(costs), vmax=vmax)
        cmap = plt.get_cmap("berlin")
        ax_grad.scatter(grad_x, grad_y, c=costs, s=50, cmap=cmap, norm=norm, alpha=0.7, linewidths=0)
        
        
        ax_grad.scatter(grad_x[idx_grad], grad_y[idx_grad],
                        s=100, edgecolor=method_to_color[name], linewidth=2.5,
                        facecolor='none', marker=method_to_marker[name], color=method_to_color[name])
  
        mark = ((pred_gain == max_gain) and underline == "max") or ((pred_gain == min_gain) and underline == "min")

        
        ax_grad.set_xticks([])
        ax_grad.set_yticks([])
        ax_grad.set_xlabel('')
        ax_grad.set_ylabel('')


        method_emb_pca = emb_pca.transform(np.vstack([embeddings_groundset, method_emb]))[-len(method_emb):]
        if use_cosine:
            dists_emb = cosine_distances(emb_reduced[:-1], method_emb_pca)
        else:
            dists_emb = np.linalg.norm(emb_reduced[:-1, None, :] - method_emb_pca[None, :, :], axis=2)
        idx_emb = np.argmin(dists_emb, axis=0)
        ax_emb = axes[i + n_cols]
        
        ax_emb.scatter(emb_x, emb_y, c=costs, s=50, cmap=cmap, norm=norm, alpha=0.7, linewidths=0)
        ax_emb.scatter(emb_x[idx_emb], emb_y[idx_emb],
                        s=100, edgecolor=method_to_color[name], linewidth=2.5,
                        facecolor='none', marker=method_to_marker[name], color=method_to_color[name])
 

        ax_emb.set_xticks([])
        ax_emb.set_yticks([])
        ax_emb.set_xlabel('')
        ax_emb.set_ylabel('')
        ax_emb.set_facecolor("#e0e0e081")

        if mark:
            ax_grad.set_title(
                r"$\mathbf{" + name + "}$" + f"\n$\\xi^{{\\mathcal{{SR}}}}$={pred_gain:.2f} dB",
                fontsize=22
            )
            ax_grad.set_facecolor("#cdcdcd")
            ax_emb.set_facecolor("#cdcdcd")
        else:
            ax_grad.set_title(
                r"$" + name + "$" + f"\n$\\xi^{{\\mathcal{{SR}}}}$={pred_gain:.2f} dB",
                fontsize=22
            )
        
        try:
            if len(idx_grad) > 2:
            
                hull = ConvexHull(np.column_stack([grad_x[idx_grad], grad_y[idx_grad]]))
                poly = Polygon(np.column_stack([grad_x[idx_grad], grad_y[idx_grad]])[hull.vertices],
                            closed=True, alpha=0.4, facecolor=method_to_color[name])
                ax_grad.add_patch(poly)

            if len(idx_emb) > 2:
                hull = ConvexHull(np.column_stack([emb_x[idx_emb], emb_y[idx_emb]]))
                poly = Polygon(np.column_stack([emb_x[idx_emb], emb_y[idx_emb]])[hull.vertices],
                            closed=True, alpha=0.4, facecolor=method_to_color[name])
                ax_emb.add_patch(poly)
        except:
            print("skipping")

       
            
    row_titles = ['LossGrad', 'Qwen3Emb']
    for row_idx, row_title in enumerate(row_titles):
        ax_title = fig.add_subplot(gs[row_idx, 0])
        ax_title.text(0.2, 0.5, row_title, va='center', ha='left',  
                    rotation=90, fontsize=20, fontweight='bold', transform=ax_title.transAxes)
        ax_title.axis('off')

      
    if costs is not None:
        vmax = np.percentile(costs, vmax_percentile)
        norm = Normalize(vmin=np.min(costs), vmax=vmax)
        cmap = plt.get_cmap("berlin")
        sm = ScalarMappable(cmap=cmap, norm=norm)
        sm.set_array(costs)
        

        cbar_ax = fig.add_subplot(gs[:, -1])
        cbar = fig.colorbar(sm, cax=cbar_ax)
        cbar.ax.tick_params(labelsize=20)

    fig.tight_layout()
    path = os.path.join(f"./figures/tsne_plots/{os.path.basename(estimator.model_path.split('_')[0])}/",f"{estimator.get_config_string().split(':')[0]}",label, f"k_{k}_idx_{test_idx}.pdf")
    os.makedirs(os.path.dirname(path),exist_ok=True)
    fig.savefig(path, format='pdf', bbox_inches='tight')
    plt.show()

In [ ]:
target =[
  {
    "content": "Find the minimum value of the algebraic expression $|\\sin x+\\cos x+\\tan x+\\cot x+\\sec x+\\csc x|$ where $x$ is a real number.",
    "role": "user"
  },
  {
    "content": "To solve the problem of finding the minimum value of the expression \\( |\\sin x + \\cos x + \\tan x + \\cot x + \\sec x + \\csc x| \\), we can follow these steps:\n\n1. **Define the Variables and the Expression:**\n    Use SymPy to define the trigonometric variables and the expression.\n\n2. **Analyze the Domain:**\n    Determine the domain of the expression. The trigonometric functions \\(\\tan x\\), \\(\\cot x\\), \\(\\sec x\\), and \\(\\csc x\\) are undefined at certain points (where their respective denominators are zero).\n\n3. **Find Critical Points:**\n    Use differentiation to find the critical points of the expression within its domain.\n\n4. **Evaluate the Expression:**\n    Evaluate the expression at the critical points and at the endpoints of the domain, if necessary, to find the minimum value.\n\nLet's implement this in Python using SymPy:\n```python\nimport sympy as sp\n\n# Define the variable\nx = sp.symbols('x')\n\n# Define the expression\nexpr = sp.sin(x) + sp.cos(x) + sp.tan(x) + sp.cot(x) + sp.sec(x) + sp.csc(x)\n\n# Find the derivative of the expression\nexpr_diff = sp.diff(expr, x)\n\n# Find the critical points by solving the derivative equal to zero\ncritical_points = sp.solve(expr_diff, x, domain=sp.Reals)\n\n# We need to filter out the points where the expression is undefined\n# The expression is undefined where tan(x), cot(x), sec(x), or csc(x) are undefined\nundefined_points = set()\nfor n in range(-10, 10):  # Considering a small range for practical purposes\n    undefined_points.add(sp.S.Pi/2 + n * sp.Pi)  # points where tan(x), sec(x), csc(x) are undefined\n    undefined_points.add(n * sp.Pi)  # points where cot(x), csc(x) are undefined\n\n# Evaluate the expression at valid critical points\nvalid_critical_points = [p for p in critical_points if p not in undefined_points]\n\n# Evaluate the expression for each valid critical point\nmin_value = float('inf')\nfor point in valid_critical_points:\n    value = abs(expr.subs(x, point))\n    if value < min_value:\n        min_value = value\n\nprint(min_value)\n```\n```output\nTimed out after 5 seconds.\n```\nThe expression \\( |\\sin x + \\cos x + \\tan x + \\cot x + \\sec x + \\csc x| \\) is quite complex and involves trigonometric functions that can be undefined at many points. Instead of attempting to find an exact symbolic solution, which might be computationally expensive, we can take a numerical approach to approximate the minimum value.\n\nHere’s how we can approach this:\n\n1. **Define the Expression:**\n    Use SymPy to define the trigonometric variables and the expression.\n\n2. **Evaluate Numerically:**\n    Evaluate the expression at a large number of points within one period of the trigonometric functions (e.g., from \\(0\\) to \\(2\\pi\\)) to approximate the minimum value.\n\nLet's implement this approach in Python using SymPy and NumPy for numerical evaluation:\n```python\nimport sympy as sp\nimport numpy as np\n\n# Define the variable\nx = sp.symbols('x')\n\n# Define the expression\nexpr = sp.sin(x) + sp.cos(x) + sp.tan(x) + sp.cot(x) + sp.sec(x) + sp.csc(x)\n\n# Convert the sympy expression to a numerical function\nexpr_func = sp.lambdify(x, expr, 'numpy')\n\n# Define the domain for evaluation\ndomain = np.linspace(0, 2 * np.pi, 10000)  # Evaluating over one period with many points\n\n# Evaluate the expression numerically\nvalues = expr_func(domain)\n\n# Filter out invalid (undefined) values\nvalues = values[np.isfinite(values)]\n\n# Find the minimum absolute value\nmin_value = np.min(np.abs(values))\n\nprint(min_value)\n```\n```output\n1.82842712818488\n```\nThe numerical approach has given us an approximate minimum value of the expression \\( |\\sin x + \\cos x + \\tan x + \\cot x + \\sec x + \\csc x| \\approx 1.82842712818488 \\).\n\nTo summarize, the minimum value of the given expression, based on our numerical method, is:\n\\[\n\\boxed{1.82842712818488}\n\\]\n\nThis result should be a good approximation due to the high resolution of numerical points used over the interval \\([0, 2\\pi]\\).",
    "role": "assistant"
  }
]
matches = test_dataset.filter(
    lambda x, idx: x["messages"] == target,
    with_indices=True
)
matches["indices"]

In [ ]:
import numpy as np
np.random.seed(42)
random_indices = [np.random.randint(0,len(test_dataset)) for _ in range (0,20)] + [np.random.randint(0,len(test_dataset)) for _ in range (0,30)]

In [ ]:
def encode(text,i):  

    emb = embedding_model.encode(
        text, convert_to_numpy=True, normalize_embeddings=True,batch_size=1,#show_progress_bar=True
    )
    torch.cuda.empty_cache()
    return emb

mmethods = []
def plot(base_explanation_class, divine_explanation_class, fl_explanation_class, indices, estimator,use_cosine=False,dim_reduction_method="tsne", label=".", underline=None):

    def get_gradients_selection(explanation):
        return torch.stack(estimator.get_gradient(train_dataset_name, train_dataset_split, explanation.documents)).to(torch.float32).numpy()

    def get_embeddings_selection(explanation):
        texts = [
            tokenizer.decode(tokenizer.apply_chat_template(messages, tokenize=True)[0:4096], skip_special_tokens=True)
            for messages in train_dataset["messages"][explanation.documents]
        ]
        return np.stack([encode(t,i) for i,t in enumerate(texts)])

    for test_idx in indices:
    # [
    #     # *random_indices,
    #     # 171,
    #     # 428,
    #     # 416, # bad examples
    #     # 97, 348, # comply not alignment
    #     # 542, # unsupported modality
    #     # 39,1, # math
    #     # 950, # unique information 
    #     # 717, # non-english non-latin script
    #     # 997, # code        
    #     ]:
        for k in [5]:#,5,10,25]:
            cache = {}
            print(f"################ Top-{k} for {test_idx}##########")
            
            base_explanation = fl_explanation_class(
                test_idx, estimator,
                train_dataset_name, train_dataset_split,
                test_dataset_name, test_dataset_split,
                k=k, m=100, keep_gradients=True
            )

         
            costs_groundset = base_explanation.costs[base_explanation.groundset_explanation.documents]
            costs_groundset = MinMaxScaler((1, 2)).fit_transform(costs_groundset.values.reshape(-1, 1)).flatten()

       
            gradients_groundset = get_gradients_selection(base_explanation.groundset_explanation)
            gradient_test_instance = estimator.get_gradient(
                test_dataset_name, test_dataset_split, test_idx
            ).to(torch.float32).numpy()

         
            torch.cuda.empty_cache()
            embeddings_groundset = get_embeddings_selection(base_explanation.groundset_explanation)
            embedding_test_instance = embedding_model.encode(
                tokenizer.decode(tokenizer.apply_chat_template(test_dataset["messages"][test_idx], tokenize=True)[0:4096], skip_special_tokens=True),
                convert_to_numpy=True, normalize_embeddings=True, batch_size=1
            )

    
            methods = []
            # naive
            naive_top_k_explanation = base_explanation_class(test_idx, estimator,
                                                                train_dataset_name, train_dataset_split,
                                                                test_dataset_name, test_dataset_split, k=k)
            grads_naive = get_gradients_selection(naive_top_k_explanation)
            methods.append((
                f"naive Top-{k}",
                naive_top_k_explanation,
                grads_naive,
                get_embeddings_selection(naive_top_k_explanation),
                estimator
            ))
           
            gradients_groundset_norm = gradients_groundset / (np.linalg.norm(
                gradients_groundset, axis=1, keepdims=True
            )+1e-10)


            gradient_test_instance_norm = gradient_test_instance / (np.linalg.norm(
                gradient_test_instance
            )+1e-10)

   
            cos_sims = gradients_groundset_norm @ gradient_test_instance_norm
                        
            # DIVINE
            divine_explanation = divine_explanation_class(test_idx, estimator,
                                                                 train_dataset_name, train_dataset_split,
                                                                 test_dataset_name, test_dataset_split,
                                                                 k=k, m=100, keep_gradients=True)
            methods.append((
                "DIVINE",
                divine_explanation,
                get_gradients_selection(divine_explanation),
                get_embeddings_selection(divine_explanation),
                estimator
            ))
            # AIDE
            AIDE_explanation = AIDE(test_idx, estimator,
                                             train_dataset_name, train_dataset_split,
                                             test_dataset_name, test_dataset_split,
                                             k=k, m=100, keep_gradients=True)
            methods.append((
                "AIDE",
                AIDE_explanation,
                get_gradients_selection(AIDE_explanation),
                get_embeddings_selection(AIDE_explanation),
                estimator
            ))
            # FL 
            for lambda_ in [0.25, 0.5, 0.75, 1]:
                fl_explanation = fl_explanation_class(test_idx, estimator,
                                                                 train_dataset_name, train_dataset_split,
                                                                 test_dataset_name, test_dataset_split,
                                                                 k=k, m=100, lambda_=lambda_, keep_gradients=True)
                methods.append((
                    f"FL λ={lambda_}",
                    fl_explanation,
                    get_gradients_selection(fl_explanation),
                    get_embeddings_selection(fl_explanation),
                    estimator
                ))
            def get_pred_gain(explanation):
                ddf = df[
                    (df["linear_coder"] == linear_coder) & 
                    (df["explanation_type"].str.startswith(explanation.description)) &
                    (df["estimator"] == estimator.get_config_string()) &
                    (df["document_idx"] == explanation.document_idx) &
                    (df["model"] == "OLMo-2-0425-1B_tulu-3-sft-olmo-2-mixture-0225_lr0.0001_seed42")
                ]
                assert len(ddf) == 1
                pred_gain = 10*np.log10(ddf.iloc[0]["pred_gain"])
                return pred_gain
            
            methods = [(name, explanation, grads, embds, estimator, get_pred_gain(explanation), label, underline) for name, explanation, grads, embds, estimator in methods]
    
            plot_single_grid(
                title=f"Test idx {test_idx} - Top-{k}",
                gradients_groundset=gradients_groundset,
                gradient_test_instance=gradient_test_instance,
                embeddings_groundset=embeddings_groundset,
                embedding_test_instance=embedding_test_instance,
                methods=methods,
                costs=costs_groundset,
                n_rows=2,
                n_cols=7,
                figsize=(24, 7),
                test_idx=test_idx,
                k=k,
                use_cosine=use_cosine,
                dim_reduction_method=dim_reduction_method,
                cache=cache
            )

            mmethods.append(methods)
            print("------------------------------------------------")
            print(estimator)
            

In [ ]:
plot(TopKMostInfluential, DIVINEMostInfluential, FacilityLocationMostInfluential, [466], estimator_datainf, use_cosine=True, dim_reduction_method="tsne") # main paper

In [ ]:
cls_dict = {
    "most helpful": [
        TopKMostHelpful,
        DIVINEMostHelpful,
        FacilityLocationMostHelpful,
    ],
    "most harmful": [
        TopKMostHarmful,
        DIVINEMostHarmful,
        FacilityLocationMostHarmful,
    ],
    "most influential": [
        TopKMostInfluential,
        DIVINEMostInfluential,
        FacilityLocationMostInfluential,
    ],
    "least influential": [
        TopKLeastInfluential,
        DIVINELeastInfluential,
        FacilityLocationLeastInfluential,
    ],
}

In [ ]:
# appendix: selections with highest- and lowest selection relevance scores for each estimator

for estimator in [estimator_datainf, estimator_less, estimator_bm25]:
    print(estimator.get_config_string())
    df_estimator = df[df["explanation_type"].str.contains("Top-5") & (df["estimator"] == estimator.get_config_string()) & (df["pred_gain"] > 0)]
    
    highest_gain = df_estimator.nlargest(1, "pred_gain")
    classes = [classes for type_, classes in cls_dict.items() if type_ in highest_gain.explanation_type.iloc[0]][0]   
    plot(*classes, highest_gain.document_idx, estimator, use_cosine=True, dim_reduction_method="tsne", label="highest_sr/"+highest_gain.explanation_type.iloc[0].replace(" ","_").replace(' ','_').replace("(", "").replace(")",""), underline="max")

    lowest_gain = df_estimator.nsmallest(1, "pred_gain")
    classes = [classes for type_, classes in cls_dict.items() if type_ in lowest_gain.explanation_type.iloc[0]][0]
    plot(*classes, lowest_gain.document_idx, estimator, use_cosine=True, dim_reduction_method="tsne", label="lowest_sr/"+lowest_gain.explanation_type.iloc[0].replace(" ","_").replace(' ','_').replace("(", "").replace(")",""), underline="min")


In [ ]:
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import matplotlib as mpl
import matplotlib.patches as patches
import re
import matplotlib.colors as mcolors
import os



fm.fontManager.addfont("fonts/NotoSansCJKtc-Regular.otf")
fm.fontManager.addfont("fonts/NotoColorEmoji-Regular.ttf")
fm.fontManager.addfont("fonts/NotoSansEthiopic-VariableFont_wdth,wght.ttf")
fm.fontManager.addfont("fonts/NotoSansTelugu-VariableFont_wdth,wght.ttf")
fm.fontManager.addfont("fonts/JetBrainsMono-Regular.ttf")
fm.fontManager.addfont("fonts/NotoSansDevanagari-VariableFont_wdth,wght.ttf")

TEXT_FONT_FAMILY = [
    "Noto Sans CJK TC",
    "Noto Color Emoji",
    "Noto Sans Ethiopic",
    "Noto Sans Telugu",
    "DejaVu Sans",
    "Noto Sans Devanagari"
]
CODE_FONT_FAMILY = "JetBrains Mono"

mpl.rcParams["font.family"] = TEXT_FONT_FAMILY
mpl.rcParams["mathtext.fontset"] = "dejavusans"

def darken_color(color, amount=0.6):
    c = mcolors.to_rgb(color)
    return tuple(amount * x for x in c)

def split_text_and_code(text):
    """
    Splits text into segments: (segment_text, is_code)
    Handles ``` code blocks with optional language.
    """
    parts = []
    pattern = re.compile(r"(`{3,}.*?\n)(.*?)(\n`{3,})", re.DOTALL)
    last = 0

    for m in pattern.finditer(text):
        # Text before the code block
        if m.start() > last:
            parts.append((text[last:m.start()], False))
        # Only the inner code
        code_content = m.group(2)
        parts.append((code_content, True))
        last = m.end()

    # Any remaining text after last code block
    if last < len(text):
        parts.append((text[last:], False))

    return parts

def draw_chat(
    ax,
    idx,
    messages,
    x0=0.02,
    y0=0.95,
    line_chars=35,
    fontsize=5,
    line_chars_cjk_or_cyrillic=10,
    line_chars_code= 30,
    onecol=False,
    add_label=False,
):
    chat_y = y0

    def split_lines(text, line_chars, line_chars_cjk_or_cyrillic,is_code,line_chars_code):
        out = []
        for line in text.splitlines():
            has_cjk_or_cyrillic = bool(re.search(r"[\u4e00-\u9fff\u0400-\u04FF]", line))
            limit = line_chars_cjk_or_cyrillic if has_cjk_or_cyrillic else (line_chars_code if is_code else line_chars)

            while len(line) > limit:
                split_pos = limit if has_cjk_or_cyrillic else max(line.rfind(" ", 0, limit), limit)
                out.append(line[:split_pos].rstrip())
                line = line[split_pos:].lstrip()
            if line:
                out.append(line)
        return out

    pre_splits = []
    for i, msg in enumerate(messages[:2]):
        # Determine max_lines
        max_lines = 3
        if i == 0:
            max_lines = 1 if (add_label and onecol) else 3
        elif i == 1:
            max_lines = 3 if (add_label and onecol) else 2

        segments = split_text_and_code(msg["content"])


        norm_segments = []
        for seg_text, is_code in segments:
            if not is_code:
                seg_text = re.sub(r"\\\((.*?)\\\)", r"$\1$", seg_text, flags=re.DOTALL)
                seg_text = re.sub(r"\\\[\s*(.*?)\s*\\\]", r"$\1$", seg_text, flags=re.DOTALL)
                seg_text = re.sub(r"^(#{1,3})\s*(.*)$", r"\2", seg_text, flags=re.MULTILINE)
            norm_segments.append((seg_text.strip("\n"), is_code))


        split_data = []
        total_lines = 0
        for seg_text, is_code in norm_segments:
            if not seg_text:
                continue
            lines = split_lines(seg_text, line_chars, line_chars_cjk_or_cyrillic,is_code,line_chars_code)
            if max_lines and total_lines + len(lines) > max_lines:
                lines = lines[: max_lines - total_lines]
                if lines:
                    lines[-1] += "..."
            total_lines += len(lines)
            split_data.append((lines, is_code))

        pre_splits.append((i, msg, max_lines, split_data))


    for i, msg, max_lines, split_data in pre_splits:
        texts = []
        cursor_y = chat_y
        total_height = 0
        max_width = 0

 
        for lines, is_code in split_data:
            wrapped = "\n".join([l.replace("****", "") for l in lines if l.replace("****", "") != ""])
            t = ax.text(
                0,
                cursor_y,
                wrapped,
                va="top",
                ha="left",
                fontsize=fontsize,
                color="white",
                fontfamily=CODE_FONT_FAMILY if is_code else None,
            )
            renderer = ax.figure.canvas.get_renderer()
            bbox = t.get_window_extent(renderer)
            h = bbox.height / ax.bbox.height
            w = bbox.width / ax.bbox.width

            total_height += h
            max_width = max(max_width, w)
            cursor_y -= h + 0.005
            texts.append(t)

        # Bubble dimensions
        bubble_height_axes = total_height + 0.02
        bubble_width_axes = max_width + 0.02

    
        color = base_colors[idx] if i % 2 == 0 else darken_color(base_colors[idx], 0.6)
        bubble_x = x0 if msg["role"] == "assistant" else 1.0 - bubble_width_axes - 0.02

        # Draw bubble
        rect = patches.FancyBboxPatch(
            (bubble_x, chat_y - bubble_height_axes),
            bubble_width_axes,
            bubble_height_axes,
            boxstyle="round,pad=0.02,rounding_size=0.02",
            fc=color,
            ec=color,
            transform=ax.transAxes,
            clip_on=False,
        )
        ax.add_patch(rect)

     
        cursor_y = chat_y - 0.005
        for t in texts:
            t.set_position((bubble_x + 0.01, cursor_y))
            renderer = ax.figure.canvas.get_renderer()
            h = t.get_window_extent(renderer).height / ax.bbox.height
            cursor_y -= h + 0.005

        # Optional label
        if add_label:
            label = "Assistant →" if msg["role"] == "assistant" else "← User"
            ha = "left" if msg["role"] == "assistant" else "right"
            lx = (
                bubble_x - (0.4 if onecol else 0.3)
                if msg["role"] == "assistant"
                else bubble_x + bubble_width_axes + (0.3 if onecol else 0.2)
            )
            ax.text(
                lx,
                chat_y - bubble_height_axes / 2,
                label,
                va="center",
                ha=ha,
                fontsize=fontsize + (3 if onecol else 5),
                fontweight="bold",
                transform=ax.transAxes,
            )

        chat_y -= bubble_height_axes + 0.06

    return chat_y

In [ ]:

def plot_chat_case_study(example_to_explain, docs, pdf_path="figures/chat_case_study_fixed.pdf",onecol=False, underline=None):
    C= 3 if onecol else 5
    page_width_pt = 219.08614 if onecol else 455.24411
    page_height_pt = 400 if onecol else 704.60031 -198.425
    page_width_in = page_width_pt / 72
    page_height_in = page_height_pt / 72

    fig = plt.figure(figsize=(page_width_in, page_height_in))
    fig.subplots_adjust(left=0.1, right=1, top=1, bottom=0)


    margin_left = 0.04
    col_spacing = 0.00
    col_width = (1 / C) * (1 - margin_left) * 0.98
    grid_width = C * col_width + (C - 1) * col_spacing
    grid_center_x = margin_left + grid_width / 2
    
    row_height = 0.08#0.12
    row_spacing = 0.00

    
    ax1_width = 0.582
    ax1_left = grid_center_x - ax1_width / 2

    ax1 = fig.add_axes([ax1_left, 0.75, ax1_width, 0.25])       
    
    chat_end =draw_chat(ax1, 0, example_to_explain, 
                        x0=0.05, 
                        y0=0.97,  
                        line_chars=58 if onecol else 78,
                        line_chars_code=40 if onecol else 57,
                        fontsize=5.5, add_label=True, onecol=onecol)


    ax1.set_xticks([])
    ax1.set_yticks([]) 
    ax1.set_frame_on(False)
    ax1_bottom_fig = ax1.get_position().y0 


    n_rows = len(docs)

    start_x = margin_left
    start_y = ax1_bottom_fig + 0.07 if onecol else ax1_bottom_fig + 0.02  # small gap

    docs = [(method_name, docs_col, pred_gain, underline) for method_name, docs_col, pred_gain,underline in docs if "FL λ=0" != method_name]
    
    pred_gains = [m[-3] for m in methods]
    min_gain = min(pred_gains)
    max_gain = max(pred_gains)
    for r, (method_name, docs_col, pred_gain, underline) in enumerate(docs): 
        n_cols = min(len(docs_col), C)
        y = start_y - r * (row_height + row_spacing)
        for c, doc in enumerate(docs_col[0:C]): 
            x = start_x + c * (col_width + col_spacing)
            ax_outer = fig.add_axes([x, y, col_width, row_height])

            pad = 0.04
            ax = ax_outer.inset_axes([pad, pad, 1 - 2 * pad, 1 - 2 * pad])
            is_left_edge = (c == 0)
            is_right_edge = (c == n_cols - 1)
            is_top_edge = (r == 0)
            is_bottom_edge = (r == n_rows - 1)

   
            ax_outer.spines['left'].set_visible(not is_left_edge)
            ax_outer.spines['right'].set_visible(not is_right_edge)
            ax_outer.spines['top'].set_visible(not is_top_edge)
            ax_outer.spines['bottom'].set_visible(not is_bottom_edge)
            for spine_name in ['left', 'right', 'top', 'bottom']:
                spine = ax_outer.spines[spine_name]
                if spine.get_visible():
                    spine.set_edgecolor("#DDDDDD")  
                    spine.set_linewidth(0.8)
            ax_outer.set_xticks([]); ax_outer.set_yticks([])
            ax.set_xticks([]); ax.set_yticks([])
            ax.set_frame_on(False)
            
            method_name_display = method_name#.replace("naive Top-5",)# "naive Top-5 (FL λ=0)")
            chat_end = draw_chat(ax, r+1,doc, 
                                 x0=0.05, 
                                 y0=0.95,
                                 line_chars=24  if onecol else 29, 
                                 line_chars_code=22  if onecol else 27,
                                 line_chars_cjk_or_cyrillic=15, 
                                 fontsize=4.5,
                                 onecol=onecol)
            ax.set_xticks([]); ax.set_yticks([])
            # print(pred_gain, max_gain, min_gain, underline)
            mark = (pred_gain == max_gain and underline == "max") or (pred_gain == min_gain and underline == "min")
            # print(mark)
            if mark:
                ax_outer.set_facecolor("#cdcdcd")
            if c == 0:
                fig.text(
                    x+0.00 if onecol else x - 0.01, 
                    y + row_height / 2,  
                    method_name_display.replace("Top-5","").replace("FL",""),
                    fontsize=7,
                    rotation=90,
                    va='center',
                    ha='right',
                    fontweight='bold' if mark else 'normal'
                )
            if r == 0:
                pass
                


    plt.show()
    with PdfPages(pdf_path) as pdf:
        pdf.savefig(fig)#, bbox_inches="tight", pad_inches=0.02)


    plt.close(fig)
for methods in mmethods:
    docs = []
    for name, explanation,_,_, estimator,pred_gain, label, underline in methods:
        docs.append((name,[(train_dataset[int(e)]["messages"]) for e in explanation.documents], pred_gain,underline))

        
    path = os.path.join(f"./figures/chat_case_study/", f"{os.path.basename(estimator.model_path.split('_')[0])}/{estimator.get_config_string().split(':')[0]}/full/",label,f"{name.replace('λ','lambda').replace(' ','_').replace('(', '').replace(')','')}_{explanation.document_idx}.pdf")
   
    os.makedirs(os.path.join(os.path.dirname(path)), exist_ok=True)
    
    plot_chat_case_study(example_to_explain=test_dataset[explanation.document_idx]["messages"], docs=docs, 
                            pdf_path=path, )
    path = path.replace("full", "onecol")
    os.makedirs(os.path.join(os.path.dirname(path)), exist_ok=True)
    plot_chat_case_study(example_to_explain=test_dataset[explanation.document_idx]["messages"], docs=docs, 
                            pdf_path=path, onecol=True, )
    
